# Movie Poster ResNet50 Embedding with Padding

- 본 노트북은 `Movie_Poster_ResNet50_Embedding.ipynb`을 베이스로 하여 진행되는 실험입니다.
- 이전 생성 노트북에선 ResNet50의 기본 center crop을 사용했습니다.
- 본 실험에서는 포스터 전체 비율을 유지하면서 남은 영역을 padding으로 채워 처리하는 실험입니다.

**확인해주세요**:
- ResNet50 기본 transform은 이미지를 224x224 중앙 crop으로 변환합니다.
    - 따라서 포스터의 상단/하단 일부가 잘립니다. 
- 포스터가 없는 영화의 처리 방식은 이후 회의에서 결정합시다.


## 1. 라이브러리 로드 및 드라이브 마운트
- 코랩 환경에서만 드라이브 마운트
- 로컬 환경일 시에도 ㄱㅊ
- 현재 런타임 cpu or gpu 여부 판단


In [ ]:
from pathlib import Path
import hashlib
import time

import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageOps
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ModuleNotFoundError:
    pass

print('torch:', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## 2. CSV 로드
- csv 파일 경로도 하이브리드로 작동합니다.
    - 1순위: 구글드라이브
    - 2순위: 로컬의 프로젝트 루트 기준
    - 3순위: 해당 노트북 상대 경로 기준

In [ ]:
csv_candidates = [
    Path('/content/gdrive/MyDrive/흥보위/sql_result/movie_snapshot_enriched_utf8_sig.csv'),
    Path('data/processed/movie_snapshot_enriched_utf8_sig.csv'),
    Path('../../data/processed/movie_snapshot_enriched_utf8_sig.csv'),
]

csv_file_path = next((path for path in csv_candidates if path.exists()), csv_candidates[0])
if csv_file_path is None:
    raise FileNotFoundError(
        'CSV 파일을 찾을 수 없습니다. csv_candidates 경로를 확인하세요.'
    )

df = pd.read_csv(csv_file_path, encoding='utf-8-sig')

print(f'CSV 경로: {csv_file_path}')
print(f'전체 행 수: {len(df):,}')
print(df.columns.tolist())
df.head()


## 2.5. 2016년 이후 데이터만 필터링


In [ ]:
min_release_date = pd.Timestamp('2016-01-01')

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
model_df = df[df['release_date'] >= min_release_date].copy()

print(f'전체 행 수: {len(df):,}')
print(f'2016년 이후 행 수: {len(model_df):,}')
print(f'제외된 행 수: {len(df) - len(model_df):,}')
model_df[['movie_name_clean', 'release_date', 'poster_url']].head()


## 3. poster_url 있는 행만 필터링


In [ ]:
valid_poster_url = model_df['poster_url'].notna() & (model_df['poster_url'].astype(str).str.strip() != '')
poster_df = model_df[valid_poster_url].copy()
poster_df = poster_df.reset_index().rename(columns={'index': 'source_row_index'})

# 처음에는 100개 정도로 테스트하고, 전체 실행할 때 None으로 바꾼다.
MAX_IMAGES = 10
if MAX_IMAGES is not None:
    poster_df = poster_df.head(MAX_IMAGES).copy()

print(f'2016년 이후 행 수: {len(model_df):,}')
print(f'poster_url 있는 행 수: {valid_poster_url.sum():,}')
print(f'이번 실행 대상 행 수: {len(poster_df):,}')
poster_df[['source_row_index', 'movie_name_clean', 'release_date', 'kmdb_doc_id', 'poster_url']].head()


## 4. 이미지 다운로드/cache


In [ ]:
def safe_image_id(row):
    if pd.notna(row.get('kmdb_doc_id')) and str(row.get('kmdb_doc_id')).strip():
        return str(row['kmdb_doc_id']).strip().replace('/', '_')

    key = f"{row.get('movie_name_clean', '')}_{row.get('release_date', '')}_{row.get('source_row_index', '')}"
    return hashlib.md5(key.encode('utf-8')).hexdigest()


base_output_dir = csv_file_path.parent

poster_dir = base_output_dir / 'posters'
poster_dir.mkdir(parents=True, exist_ok=True)

output_base_dir = base_output_dir / 'poster_resnet50_padding'
output_base_dir.mkdir(parents=True, exist_ok=True)

poster_df['poster_image_id'] = poster_df.apply(safe_image_id, axis=1)
poster_df['poster_download_url'] = poster_df['poster_url'].astype(str).str.strip()
poster_df['poster_path'] = poster_df['poster_image_id'].apply(lambda image_id: poster_dir / f'{image_id}.jpg')

print(f'결과 저장 폴더: {output_base_dir}')
print(f'포스터 저장 폴더: {poster_dir}')
poster_df[['movie_name_clean', 'poster_download_url', 'poster_path']].head()


In [ ]:
def download_image(url, output_path, timeout=15, sleep_sec=0.1):
    output_path = Path(output_path)
    if output_path.exists() and output_path.stat().st_size > 0:
        return 'cached'

    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()

        content_type = response.headers.get('Content-Type', '')
        if 'image' not in content_type.lower():
            return f'failed: non-image content-type={content_type}'

        output_path.write_bytes(response.content)
        time.sleep(sleep_sec)
        return 'downloaded'
    except Exception as exc:
        return f'failed: {type(exc).__name__}: {exc}'


download_results = []
for _, row in tqdm(poster_df.iterrows(), total=len(poster_df)):
    status = download_image(row['poster_download_url'], row['poster_path'])
    download_results.append(status)

poster_df['download_status'] = download_results
poster_df['download_success'] = poster_df['download_status'].isin(['cached', 'downloaded'])

poster_df['download_status'].value_counts().head(20)


## 5. 다운로드 성공/실패 확인


In [ ]:
success_df = poster_df[poster_df['download_success']].copy()
failed_df = poster_df[~poster_df['download_success']].copy()

print(f'다운로드 성공: {len(success_df):,}')
print(f'다운로드 실패: {len(failed_df):,}')

failed_df[['movie_name_clean', 'release_date', 'poster_download_url', 'download_status']].head(20)


## 6. 사전학습 CNN 모델 로드


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # cpu냐 gpu냐 그것이 문제로다

weights = models.ResNet50_Weights.DEFAULT # ResNet50 사전 기본 가중치 사용
cnn_model = models.resnet50(weights=weights)
cnn_model.fc = nn.Identity()  # 마지막 분류층을 제거하고 2048차원 embedding만 사용
cnn_model = cnn_model.to(device)
cnn_model.eval()

print('모델: ResNet50 pretrained')
print('embedding 차원: 2048')
print('device:', device)


## 7. 이미지 전처리
- 이미지 크기 조정
- 중앙 crop
- tensor 변환
- 정규화

In [ ]:
class PosterPaddingTransform:
    def __init__(self, image_size=224):
        self.image_size = image_size
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        self.padding_color = tuple((self.mean * 255).round().astype(np.uint8).tolist())

    def __call__(self, image):
        image = image.convert('RGB')
        resample = Image.Resampling.BICUBIC if hasattr(Image, 'Resampling') else Image.BICUBIC

        resized = ImageOps.contain(
            image,
            (self.image_size, self.image_size),
            method=resample,
        )

        padded = Image.new('RGB', (self.image_size, self.image_size), self.padding_color)
        left = (self.image_size - resized.width) // 2
        top = (self.image_size - resized.height) // 2
        padded.paste(resized, (left, top))

        array = np.asarray(padded).astype(np.float32) / 255.0
        array = (array - self.mean) / self.std
        array = np.transpose(array, (2, 0, 1))
        return torch.from_numpy(array)


image_transform = PosterPaddingTransform(image_size=224)

class PosterDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row['poster_path']).convert('RGB')
        image_tensor = self.transform(image)
        return image_tensor, idx


poster_dataset = PosterDataset(success_df, image_transform)
poster_loader = DataLoader(
    poster_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

print(f'Embedding 추출 대상 이미지 수: {len(poster_dataset):,}')


## 7.5 이미지 전처리 결과 확인

In [ ]:
import matplotlib.pyplot as plt

def denormalize_image(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    image = tensor.cpu() * std + mean
    image = image.clamp(0, 1)
    image = image.permute(1, 2, 0).numpy()
    return image


sample_count = 5

fig, axes = plt.subplots(sample_count, 2, figsize=(8, sample_count * 4))

for i in range(sample_count):
    row = success_df.iloc[i]

    original_image = Image.open(row['poster_path']).convert('RGB')
    transformed_tensor = image_transform(original_image)
    transformed_image = denormalize_image(transformed_tensor)

    axes[i, 0].imshow(original_image)
    axes[i, 0].set_title(f"Original\n{row['movie_name_clean']}")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(transformed_image)
    axes[i, 1].set_title("After padding preprocess\n224 x 224")
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

## 8. CNN embedding 추출


In [ ]:
all_embeddings = []
all_indices = []

with torch.no_grad():
    for images, indices in tqdm(poster_loader):
        images = images.to(device)
        embeddings = cnn_model(images).cpu().numpy()
        all_embeddings.append(embeddings)
        all_indices.extend(indices.numpy().tolist())

poster_embeddings = np.vstack(all_embeddings) if all_embeddings else np.empty((0, 2048), dtype=np.float32)
embedding_index_df = success_df.iloc[all_indices].reset_index(drop=True)

print('embedding shape:', poster_embeddings.shape)
embedding_index_df[['source_row_index', 'movie_name_clean', 'release_date', 'kmdb_doc_id', 'poster_path']].head()


## 9. embedding 저장


In [ ]:
embedding_npy_path = output_base_dir / 'poster_embeddings_resnet50_padding.npy'
embedding_index_path = output_base_dir / 'poster_embeddings_resnet50_padding_index.csv'
download_log_path = output_base_dir / 'poster_download_log.csv'

index_columns = [
    'source_row_index',
    'movie_name_clean',
    'release_date',
    'poster_path',
]
embedding_index_save_df = embedding_index_df[index_columns].copy()

np.save(embedding_npy_path, poster_embeddings)
embedding_index_save_df.to_csv(embedding_index_path, index=False, encoding='utf-8-sig')
poster_df.to_csv(download_log_path, index=False, encoding='utf-8-sig')

print(f'embedding 저장: {embedding_npy_path}')
print(f'index 저장: {embedding_index_path}')
print(f'download log 저장: {download_log_path}')
print(f'index 컬럼: {embedding_index_save_df.columns.tolist()}')


## 10. 저장 결과 검증


In [ ]:
loaded_embeddings = np.load(embedding_npy_path)
loaded_index_df = pd.read_csv(embedding_index_path, encoding='utf-8-sig')

print('loaded embedding shape:', loaded_embeddings.shape)
print('loaded index rows:', len(loaded_index_df))
print('shape/index match:', loaded_embeddings.shape[0] == len(loaded_index_df))
print('index columns:', loaded_index_df.columns.tolist())

loaded_index_df[['source_row_index', 'movie_name_clean', 'release_date', 'poster_path']].head()
